In [ ]:
!pip install imbalanced-learn -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, roc_auc_score, roc_curve
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
sns.set_style("whitegrid")

In [ ]:
df = pd.read_csv("creditcard.csv")
print(df.shape)
df.head()

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [ ]:
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print(class_counts)
print()
print(class_pct.round(3))

Class
0    284315
1       492
Name: count, dtype: int64

Class
0    99.827
1     0.173
Name: proportion, dtype: float64


In [ ]:
df.isnull().sum().sum()

np.int64(0)

In [ ]:
#test split
X = df.drop(columns=['Class'])
y = df['Class']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape, "Fraud rate:", y_train.mean().round(5))
print("Test shape :", X_test.shape, "Fraud rate:", y_test.mean().round(5))

Train shape: (227845, 30) Fraud rate: 0.00173
Test shape : (56962, 30) Fraud rate: 0.00172


In [ ]:
#imblearn Pipelines
lr_pipeline = ImbPipeline(steps=[
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

In [ ]:
#Random Forest Pipeline
rf_pipeline = ImbPipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
])

In [ ]:
#Hyperparameter Tunning
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
lr_param_grid = {
    'smote__k_neighbors': [5],
    'classifier__C': [0.1, 1.0, 10]
}

lr_grid = GridSearchCV(
    lr_pipeline,
    param_grid=lr_param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    verbose=2
)

lr_grid.fit(X_train, y_train)
print("Best LR params:", lr_grid.best_params_)
print("Best LR CV ROC-AUC:", lr_grid.best_score_)

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best LR params: {'classifier__C': 0.1, 'smote__k_neighbors': 5}
Best LR CV ROC-AUC: 0.9806193896825396


In [ ]:
rf_param_grid = {
    'smote__k_neighbors': [5],
    'classifier__n_estimators': [50],
    'classifier__max_depth': [10]
}

rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid=rf_param_grid,
    scoring='roc_auc',
    cv=2,
    n_jobs=-1,
    verbose=3
)

rf_grid.fit(X_train, y_train)
print("Best RF params:", rf_grid.best_params_)
print("Best RF CV ROC-AUC:", rf_grid.best_score_)

Fitting 2 folds for each of 1 candidates, totalling 2 fits


In [ ]:
def evaluate_model(name, fitted_grid, X_test, y_test):
    best_model = fitted_grid.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    print(f"===== {name} =====")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-score  : {f1:.4f}")
    print(f"ROC-AUC   : {roc_auc:.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Fraud'])
    disp.plot(cmap='Blues')
    plt.title(f"Confusion Matrix — {name}")
    plt.show()

    return {'name': name, 'precision': precision, 'recall': recall, 'f1': f1, 'roc_auc': roc_auc, 'y_proba': y_proba}

In [ ]:
lr_results = evaluate_model("Logistic Regression", lr_grid, X_test, y_test)

In [ ]:
rf_results = evaluate_model("Random Forest", rf_grid, X_test, y_test)

In [ ]:
plt.figure(figsize=(6,6))

for res in [lr_results, rf_results]:
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    plt.plot(fpr, tpr, label=f"{res['name']} (AUC = {res['roc_auc']:.3f})")

plt.plot([0,1], [0,1], linestyle='--', color='gray', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()

In [ ]:
summary = pd.DataFrame([
    {k: v for k, v in lr_results.items() if k != 'y_proba'},
    {k: v for k, v in rf_results.items() if k != 'y_proba'}
]).set_index('name')

summary.round(4)